In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("../data/ecommerce.db")
conn = sqlite3.connect(db_path)

def run_sql(query):
    return pd.read_sql(query, conn)

In [2]:
conn.executescript("""

DROP TABLE IF EXISTS fact_sales;
DROP TABLE IF EXISTS dim_customer;
DROP TABLE IF EXISTS dim_product;
DROP TABLE IF EXISTS dim_seller;
DROP TABLE IF EXISTS dim_date;

""")

conn.commit()

print("Warehouse cleaned.")

Warehouse cleaned.


In [3]:
conn.executescript("""

DROP TABLE IF EXISTS fact_sales;

CREATE TABLE fact_sales AS

SELECT

    oi.order_id,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,

    o.customer_id,

    DATE(o.order_purchase_timestamp) AS purchase_date,

    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,

    o.order_status,

    oi.price,
    oi.freight_value,

    (oi.price + oi.freight_value) AS gross_value

FROM order_items oi

INNER JOIN orders o
ON oi.order_id = o.order_id;

""")

conn.commit()

print("✅ Base fact_sales table created.")

✅ Base fact_sales table created.


In [4]:
run_sql("""
SELECT COUNT(*) AS total_rows
FROM fact_sales;
""")

,total_rows
0,112650


In [5]:
run_sql("""
SELECT

COUNT(*) AS total_rows,

COUNT(DISTINCT order_id || '-' || order_item_id) AS unique_rows

FROM fact_sales;
""")

,total_rows,unique_rows
0,112650,112650


In [6]:
run_sql("""
SELECT *
FROM fact_sales
LIMIT 5;
""")

,order_id,order_item_id,product_id,seller_id,customer_id,purchase_date,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_status,price,freight_value,gross_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,3ce436f183e68e07877b285a838db11a,2017-09-13,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29 00:00:00,delivered,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,f6dd3ec061db4e3987629fe6b26e5cce,2017-04-26,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15 00:00:00,delivered,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,6489ae5e4333f3693df5ad4372dab6d3,2018-01-14,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05 00:00:00,delivered,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,d4eb9395c8c0431ee92fce09860c5a06,2018-08-08,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20 00:00:00,delivered,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,58dbd0b2d70206bf40e62cd34e84d795,2017-02-04,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17 00:00:00,delivered,199.90,18.14,218.04


In [7]:
conn.executescript("""

ALTER TABLE fact_sales
ADD COLUMN customer_unique_id TEXT;

ALTER TABLE fact_sales
ADD COLUMN customer_city TEXT;

ALTER TABLE fact_sales
ADD COLUMN customer_state TEXT;

""")

conn.commit()

print("✅ Customer columns added.")

✅ Customer columns added.


In [8]:
conn.execute("DROP TABLE IF EXISTS fact_sales;")
conn.commit()

In [9]:
conn.executescript("""

CREATE TABLE fact_sales AS

SELECT

    oi.order_id,
    oi.order_item_id,

    oi.product_id,
    oi.seller_id,

    o.customer_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,

    DATE(o.order_purchase_timestamp) AS purchase_date,

    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,

    o.order_status,

    oi.price,
    oi.freight_value,

    (oi.price + oi.freight_value) AS gross_value

FROM order_items oi

INNER JOIN orders o

ON oi.order_id = o.order_id

INNER JOIN customers c

ON o.customer_id = c.customer_id;

""")

conn.commit()

print("✅ fact_sales rebuilt")

✅ fact_sales rebuilt


In [10]:
run_sql("""
SELECT COUNT(*)
FROM fact_sales;
""")

,COUNT(*)
0,112650


In [11]:
conn.executescript("""

DROP TABLE IF EXISTS order_payment_summary;

CREATE TABLE order_payment_summary AS

SELECT

    order_id,

    SUM(payment_value) AS total_payment,

    COUNT(*) AS payment_records,

    GROUP_CONCAT(DISTINCT payment_type) AS payment_methods,

    MAX(payment_installments) AS max_installments

FROM payments

GROUP BY order_id;

""")

conn.commit()

print("✅ order_payment_summary created.")

✅ order_payment_summary created.


In [12]:
run_sql("""
SELECT *
FROM order_payment_summary
LIMIT 5;
""")

,order_id,total_payment,payment_records,payment_methods,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,credit_card,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,credit_card,3
2,000229ec398224ef6ca0657da4fc703e,216.87,1,credit_card,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,credit_card,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,credit_card,3


In [13]:
run_sql("""
SELECT COUNT(*)
FROM order_payment_summary;
""")

,COUNT(*)
0,99440


In [14]:
conn.executescript("""

DROP TABLE IF EXISTS fact_sales_v2;

CREATE TABLE fact_sales_v2 AS

SELECT

    fs.*,

    ops.total_payment,

    ops.payment_records,

    ops.payment_methods,

    ops.max_installments

FROM fact_sales fs

LEFT JOIN order_payment_summary ops

ON fs.order_id = ops.order_id;

""")

conn.commit()

print("✅ fact_sales_v2 created.")

✅ fact_sales_v2 created.


In [15]:
run_sql("""
SELECT COUNT(*)
FROM fact_sales_v2;
""")

,COUNT(*)
0,112650


In [16]:
run_sql("""
SELECT COUNT(*) AS missing_payment
FROM fact_sales_v2
WHERE total_payment IS NULL;
""")

,missing_payment
0,3


In [17]:
conn.executescript("""

DROP TABLE IF EXISTS order_review_summary;

CREATE TABLE order_review_summary AS

SELECT

    r.order_id,
    r.review_score,
    r.review_creation_date,
    r.review_answer_timestamp

FROM reviews r

INNER JOIN (

    SELECT

        order_id,
        MAX(review_creation_date) AS latest_review

    FROM reviews

    GROUP BY order_id

) latest

ON r.order_id = latest.order_id

AND r.review_creation_date = latest.latest_review;

""")

conn.commit()

print("✅ order_review_summary created.")

✅ order_review_summary created.


In [18]:
run_sql("""
SELECT *
FROM order_review_summary
LIMIT 5;
""")

,order_id,review_score,review_creation_date,review_answer_timestamp
0,73fc7af87114b39712e6da79b0a377eb,4,2018-01-18 00:00:00,2018-01-18 21:46:59
1,a548910a1c6147796b98fdf73dbeba33,5,2018-03-10 00:00:00,2018-03-11 03:05:13
2,f9e4b658b201a9f2ecdecbb34bed034b,5,2018-02-17 00:00:00,2018-02-18 14:36:24
3,658677c97b385a9be170737859d3511b,5,2017-04-21 00:00:00,2017-04-21 22:02:06
4,8e6bfb81e283fa7e4f11123a3fb894f1,5,2018-03-01 00:00:00,2018-03-02 10:26:53


In [19]:
run_sql("""
SELECT COUNT(*)
FROM order_review_summary;
""")

,COUNT(*)
0,98829


In [20]:
run_sql("""
SELECT

COUNT(*) total_rows,

COUNT(DISTINCT order_id) unique_orders

FROM order_review_summary;
""")

,total_rows,unique_orders
0,98829,98673


In [21]:
conn.executescript("""

DROP TABLE IF EXISTS fact_sales_v3;

CREATE TABLE fact_sales_v3 AS

SELECT

    fs.*,

    ors.review_score

FROM fact_sales_v2 fs

LEFT JOIN order_review_summary ors

ON fs.order_id = ors.order_id;

""")

conn.commit()

print("✅ fact_sales_v3 created.")

✅ fact_sales_v3 created.


In [22]:
run_sql("""
SELECT COUNT(*)
FROM fact_sales_v3;
""")

,COUNT(*)
0,112834


In [23]:
run_sql("""
SELECT COUNT(*) AS missing_reviews
FROM fact_sales_v3
WHERE review_score IS NULL;
""")

,missing_reviews
0,942


In [24]:
conn.executescript("""

DROP TABLE IF EXISTS fact_sales_final;

CREATE TABLE fact_sales_final AS

SELECT

    fs.*,

    ROUND(
        JULIANDAY(order_delivered_customer_date) -
        JULIANDAY(order_purchase_timestamp),
        2
    ) AS delivery_days,

    ROUND(
        JULIANDAY(order_approved_at) -
        JULIANDAY(order_purchase_timestamp),
        2
    ) AS approval_days,

    ROUND(
        JULIANDAY(order_delivered_carrier_date) -
        JULIANDAY(order_approved_at),
        2
    ) AS seller_processing_days,

    ROUND(
        JULIANDAY(order_delivered_customer_date) -
        JULIANDAY(order_delivered_carrier_date),
        2
    ) AS shipping_days,

    CASE

        WHEN order_delivered_customer_date
             <= order_estimated_delivery_date

        THEN 0

        ELSE 1

    END AS is_late

FROM fact_sales_v3 fs;

""")

conn.commit()

print("✅ fact_sales_final created")

✅ fact_sales_final created


In [25]:
run_sql("""
SELECT COUNT(*)
FROM fact_sales_final;
""")

,COUNT(*)
0,112834


In [26]:
run_sql("""
SELECT

delivery_days,
approval_days,
seller_processing_days,
shipping_days,
is_late

FROM fact_sales_final

LIMIT 10;
""")

,delivery_days,approval_days,seller_processing_days,shipping_days,is_late
0,7.61,0.03,6.37,1.21,0
1,16.22,0.01,8.15,8.06,0
2,7.95,0.01,1.91,6.03,0
3,6.15,0.01,2.14,4.00,0
4,25.11,0.01,11.82,13.29,0
5,6.67,1.26,0.30,5.11,0
6,8.42,0.01,1.54,6.87,0
7,5.08,1.19,-0.18,4.08,0
8,9.98,1.00,7.25,1.74,1
9,2.15,0.01,1.01,1.13,0


In [27]:
conn.executescript("""

DROP TABLE IF EXISTS order_totals;

CREATE TABLE order_totals AS

SELECT

    order_id,

    SUM(price) AS total_merchandise

FROM fact_sales_final

GROUP BY order_id;

""")

conn.commit()

In [28]:
conn.execute("""

ALTER TABLE fact_sales_final

ADD COLUMN allocated_payment REAL;

""")

conn.commit()

In [29]:
conn.executescript("""

DROP TABLE IF EXISTS fact_sales_final_v2;

CREATE TABLE fact_sales_final_v2 AS

SELECT

    fs.*,

    ROUND(
        (fs.price * 1.0 / ot.total_merchandise)
        * fs.total_payment,
        2
    ) AS allocated_payment

FROM fact_sales_final fs

LEFT JOIN order_totals ot
ON fs.order_id = ot.order_id;

""")

conn.commit()

print("✅ fact_sales_final_v2 created")

✅ fact_sales_final_v2 created


In [30]:
run_sql("""
SELECT

ROUND(SUM(allocated_payment),2) AS allocated_total

FROM fact_sales_final_v2;
""")

,allocated_total
0,None


In [31]:
run_sql("""
SELECT

ROUND(SUM(payment_value),2) AS source_total

FROM payments;
""")

,source_total
0,16008872.12


In [32]:
conn.executescript("""

DROP TABLE IF EXISTS dim_customer;

CREATE TABLE dim_customer AS

SELECT DISTINCT

customer_unique_id,
customer_city,
customer_state

FROM fact_sales_final_v2;

""")

conn.commit()

print("✅ dim_customer created")

✅ dim_customer created


In [33]:
run_sql("""
SELECT COUNT(*)
FROM dim_customer;
""")

,COUNT(*)
0,95539


In [34]:
conn.executescript("""

DROP TABLE IF EXISTS dim_product;

CREATE TABLE dim_product AS

SELECT DISTINCT

p.product_id,

COALESCE(ct.product_category_name_english,
         p.product_category_name,
         'Unknown') AS category,

p.product_weight_g,
p.product_length_cm,
p.product_height_cm,
p.product_width_cm

FROM products p

LEFT JOIN category_translation ct

ON p.product_category_name=ct.product_category_name;

""")

conn.commit()

print("✅ dim_product created")

✅ dim_product created


In [35]:
conn.executescript("""

DROP TABLE IF EXISTS dim_seller;

CREATE TABLE dim_seller AS

SELECT DISTINCT

seller_id,
seller_city,
seller_state

FROM sellers;

""")

conn.commit()

print("✅ dim_seller created")

✅ dim_seller created
